In [ ]:
# Paste your Hugging Face read token here.
HF_TOKEN = "PASTE_YOUR_HF_TOKEN_HERE"

### This Notebook: Continued Pretraining (CPT) on Business Central AL Code

This notebook is configured for **continued pretraining** of Gemma 4 E2B-it on Microsoft Dynamics 365 Business Central AL source code on Modal.

**Key differences from instruction fine-tuning:**
- Uses raw text completion (next-token prediction), not chat format
- Trains on ALL tokens (no masking of user vs assistant parts)
- Includes `embed_tokens` and `lm_head` in LoRA for domain adaptation
- Uses `UNSLOTH_RETURN_LOGITS=1` to disable CCE (not supported for CPT)
- Higher LoRA rank (128) and rank-stabilized LoRA for better adaptation
- EOS tokens added to help the model learn proper stopping

**Data source:** `business_central_al_files.jsonl` or `business_central_al_training_text.txt`, generated beforehand from `.al` files with `scripts/convert_al_to_text.py`, then loaded from a Hugging Face dataset repo, the Modal notebook filesystem, or a Modal Volume.

---

### Installation

In [ ]:
try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
except: _numpy = "numpy"; _pil = "pillow"

# Set environment variable to disable CCE for continued pretraining.
# CCE (Constant Cross Entropy) is not supported for CPT.
import os
import subprocess
import sys
os.environ["UNSLOTH_RETURN_LOGITS"] = "1"

packages = [
    "torch>=2.8.0",
    "triton>=3.4.0",
    _numpy,
    _pil,
    "torchvision",
    "bitsandbytes",
    "datasets",
    "huggingface_hub",
    "trl",
    "unsloth",
    "unsloth_zoo>=2026.4.6",
    "transformers==5.5.0",
    "torchcodec",
    "timm",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-qqq", *packages])

### Unsloth

`FastModel` supports loading nearly any model now! This includes Vision and Text models!

In [ ]:
import os
from unsloth import FastModel
import torch

HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN") or globals().get("HF_TOKEN")
if HF_TOKEN == "PASTE_YOUR_HF_TOKEN_HERE":
    HF_TOKEN = None
MODEL_NAME = "unsloth/gemma-4-E2B-it"
MAX_SEQ_LENGTH = 8192

gemma4_models = [
    # Gemma-4 instruct models:
    "unsloth/gemma-4-E2B-it",
    "unsloth/gemma-4-E4B-it",
    "unsloth/gemma-4-31B-it",
    "unsloth/gemma-4-26B-A4B-it",
    # Gemma-4 base models:
    "unsloth/gemma-4-E2B",
    "unsloth/gemma-4-E4B",
    "unsloth/gemma-4-31B",
    "unsloth/gemma-4-26B-A4B",
]

model, tokenizer = FastModel.from_pretrained(
    model_name = MODEL_NAME,
    dtype = None, # None for auto detection
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
    full_finetuning = False,
    token = HF_TOKEN, # HF Token for gated models
    device_map = {"": torch.cuda.current_device()},
)

In [ ]:
from transformers import TextStreamer

# Helper function for chat inference (for testing before CPT)
def do_gemma_4_chat_inference(messages, max_new_tokens = 128):
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt = True,
        tokenize = True,
        return_dict = True,
        return_tensors = "pt",
    ).to("cuda")

    _ = model.generate(
        **inputs,
        max_new_tokens = max_new_tokens,
        use_cache = True,
        temperature = 1.0,
        top_p = 0.95,
        top_k = 64,
        streamer = TextStreamer(tokenizer, skip_prompt = True, skip_special_tokens = True),
    )


def _get_text_tokenizer():
    return getattr(tokenizer, "tokenizer", tokenizer)


def _unused_token_ids(text_tokenizer, limit=256):
    ids = []
    for index in range(limit):
        token = f"<unused{index}>"
        token_id = text_tokenizer.convert_tokens_to_ids(token)
        if token_id is not None and token_id != text_tokenizer.unk_token_id:
            ids.append([token_id])
    return ids


# Helper function for continued pretraining inference (text completion)
def do_gemma_4_text_completion(prompt, max_new_tokens=256, temperature=0.2):
    """For CPT inference - direct text completion tuned for code-like output."""
    import torch
    from transformers import TextIteratorStreamer
    from threading import Thread

    device = "cuda" if torch.cuda.is_available() else "cpu"
    text_tokenizer = _get_text_tokenizer()

    inputs = text_tokenizer(prompt, return_tensors="pt", add_special_tokens=True).to(device)
    streamer = TextIteratorStreamer(text_tokenizer, skip_prompt=True, skip_special_tokens=True)

    do_sample = temperature is not None and temperature > 0
    generation_kwargs = dict(
        **inputs,
        streamer=streamer,
        max_new_tokens=max_new_tokens,
        use_cache=True,
        do_sample=do_sample,
        temperature=temperature if do_sample else None,
        top_p=0.9 if do_sample else None,
        top_k=40 if do_sample else None,
        repetition_penalty=1.05,
        eos_token_id=text_tokenizer.eos_token_id,
        pad_token_id=text_tokenizer.pad_token_id or text_tokenizer.eos_token_id,
        bad_words_ids=_unused_token_ids(text_tokenizer),
    )
    generation_kwargs = {key: value for key, value in generation_kwargs.items() if value is not None}

    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    generated_text = ""
    for text in streamer:
        print(text, end="", flush=True)
        generated_text += text
    thread.join()
    print()
    return generated_text


# Let's finetune Gemma 4!

You can finetune the vision and text parts for now through selection - the audio part can also be finetuned - we're working to make it selectable as well!

We now add LoRA adapters so we only need to update a small amount of parameters!

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,  # Turn off for text CPT
    finetune_language_layers   = True,   # Keep on for continued pretraining
    finetune_attention_modules = True,   # Good for CPT
    finetune_mlp_modules       = True,   # Keep on always

    # Embeddings and lm_head help domain adaptation, but they add trainable params.
    # Keep them enabled for quality; disable only if you need an even faster smoke run.
    finetune_embedding_modules = True,
    finetune_lm_head          = True,

    r = 128,           # Stronger CPT adapter; use 64 for faster smoke runs.
    lora_alpha = 64,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = True,
)

### Data Prep for Business Central Continued Pretraining

For continued pretraining, we load the generated Business Central AL training text and train the model to predict the next token. We don't use chat templates - just raw AL text chunks.

Load file-level AL training data from a Hugging Face dataset repo, the Modal notebook filesystem, or an attached Modal Volume, tokenize it, and create overlapping chunks for training.

In [ ]:
import json
import os
import random
import re
from collections import Counter
from pathlib import Path
from datasets import Dataset
from huggingface_hub import hf_hub_download

max_seq_length = MAX_SEQ_LENGTH

# Modal usage:
# 1. Upload business_central_al_files.jsonl or business_central_al_training_text.txt to a Hugging Face dataset repo.
# 2. Set HF_DATASET_REPO_ID to that repo, or provide it through a Modal Secret/env var.
# 3. Set HF_DATASET_FILENAME if the file is not at one of the default paths below.
HF_DATASET_REPO_ID = os.environ.get("HF_DATASET_REPO_ID") or "Capitaller/bc-training-data"
HF_DATASET_FILENAME = os.environ.get("HF_DATASET_FILENAME") or "business_central_al_files.jsonl"
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN") or globals().get("HF_TOKEN")
if HF_TOKEN == "PASTE_YOUR_HF_TOKEN_HERE":
    HF_TOKEN = None
MODAL_TRAINING_TEXT_PATH = os.environ.get("MODAL_TRAINING_TEXT_PATH") or None  # Optional local fallback.

training_text_path = None

if HF_DATASET_REPO_ID and not HF_DATASET_REPO_ID.startswith("YOUR_HF_USER/"):
    hf_filenames = []
    if HF_DATASET_FILENAME:
        hf_filenames.append(HF_DATASET_FILENAME)
    hf_filenames.extend([
        "business_central_al_files.jsonl",
        "baseapp-al-corpus/business_central_al_files.jsonl",
        "baseapp-al-corpus/business_central_al_training_text.txt",
        "business_central_al_training_text.txt",
        "baseapp-al-corpus/train.txt",
        "train.txt",
    ])

    last_download_error = None
    for hf_filename in dict.fromkeys(hf_filenames):
        try:
            training_text_path = Path(hf_hub_download(
                repo_id=HF_DATASET_REPO_ID,
                filename=hf_filename,
                repo_type="dataset",
                token=HF_TOKEN,
            ))
            print(f"Downloaded training text from Hugging Face: {HF_DATASET_REPO_ID}/{hf_filename}")
            break
        except Exception as error:
            last_download_error = error
    else:
        raise FileNotFoundError(
            f"Could not download training text from Hugging Face dataset {HF_DATASET_REPO_ID!r}. "
            "Set HF_DATASET_FILENAME to the file path inside the dataset repo."
        ) from last_download_error

if training_text_path is None and MODAL_TRAINING_TEXT_PATH:
    training_text_path = Path(MODAL_TRAINING_TEXT_PATH)

if training_text_path is None:
    training_text_filenames = ["business_central_al_files.jsonl", "business_central_al_training_text.txt"]
    candidate_paths = []
    for training_text_filename in training_text_filenames:
        candidate_paths.extend([
            Path.cwd() / training_text_filename,
            Path.cwd() / "data" / "bc_al_text" / training_text_filename,
            Path("/mnt") / training_text_filename,
        ])
    training_text_candidates = [path for path in candidate_paths if path.exists()]
    if not training_text_candidates and (Path.cwd() / "data").exists():
        training_text_candidates = [path for filename in training_text_filenames for path in sorted((Path.cwd() / "data").glob(f"**/{filename}"))]
    if not training_text_candidates and Path("/mnt").exists():
        training_text_candidates = [path for filename in training_text_filenames for path in sorted(Path("/mnt").glob(f"**/{filename}"))]
    if not training_text_candidates:
        training_text_candidates = [path for path in [Path.cwd() / "corpus.txt", Path("/mnt") / "corpus.txt"] if path.exists()]
    training_text_path = training_text_candidates[0] if training_text_candidates else None

if training_text_path is None or not training_text_path.exists():
    raise FileNotFoundError(
        "Set HF_DATASET_REPO_ID to your Hugging Face dataset repo, for example 'YOUR_HF_USER/bc-training-data'. "
        "If needed, also set HF_DATASET_FILENAME to the file path inside that dataset repo."
    )

raw_text = training_text_path.read_text(encoding="utf-8")
backend_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)
EOS_TOKEN = tokenizer.eos_token

FILE_START_TOKEN = "<|bc_al_file|>"
FILE_END_TOKEN = "<|end_bc_al_file|>"
AL_SECTION_START_RE = re.compile(
    r"^\s*(?:(?:local|internal|protected|procedure|trigger)\s+)?(?:procedure|trigger)\b|"
    r"^\s*(?:field|action|dataitem|column|value)\s*\(",
    re.IGNORECASE,
)
AL_ATTRIBUTE_RE = re.compile(r"^\s*\[.*\]\s*$")
CHUNK_TOKENS = max_seq_length - 1
OVERLAP_TOKENS = 256
MIN_CHUNK_TOKENS = 256
VALIDATION_FILE_RATIO = 0.02
RANDOM_SEED = 3407

if not EOS_TOKEN:
    raise ValueError("Tokenizer does not define an EOS token.")

def parse_al_file_records(corpus_text):
    records = []
    for block in corpus_text.split(FILE_START_TOKEN):
        block = block.strip("\n")
        if not block or "\n" not in block:
            continue

        source, rest = block.split("\n", 1)
        content, _, _ = rest.partition(FILE_END_TOKEN)
        content = content.strip()
        if content:
            records.append({"source": source.strip(), "content": content})

    if records:
        return records

    fallback_text = corpus_text.strip()
    if not fallback_text:
        raise ValueError("Training corpus is empty.")
    return [{"source": training_text_path.name, "content": fallback_text}]

def parse_file_jsonl_records(jsonl_text):
    records = []
    for line_number, line in enumerate(jsonl_text.splitlines(), start=1):
        if not line.strip():
            continue

        row = json.loads(line)
        row_text = row.get("text", "").strip()
        if row_text.startswith(FILE_START_TOKEN):
            records.extend(parse_al_file_records(row_text))
            continue

        content = (row.get("content") or row_text).strip()
        source = (row.get("source") or f"row-{line_number}.al").strip()
        if content:
            records.append({"source": source, "content": content})

    if not records:
        raise ValueError(f"No AL file records found in JSONL file: {training_text_path}")
    return records

def format_file_record(record):
    return f"{FILE_START_TOKEN}{record['source']}\n{record['content']}\n{FILE_END_TOKEN}"

def token_count(text):
    return len(backend_tokenizer.encode(text, add_special_tokens=False))

def chunk_text(record, body_text, is_file_end=False, chunk_kind="section_pack"):
    text = f"{FILE_START_TOKEN}{record['source']}\n{body_text.strip()}"
    if is_file_end:
        text += f"\n{FILE_END_TOKEN}{EOS_TOKEN}"
    return {
        "text": text,
        "source": record["source"],
        "is_file_end": is_file_end,
        "chunk_kind": chunk_kind,
    }

def split_al_sections(content):
    lines = content.splitlines(keepends=True)
    if not lines:
        return []

    starts = [0]
    for index, line in enumerate(lines):
        if not AL_SECTION_START_RE.match(line):
            continue

        start = index
        while start > starts[-1] and (not lines[start - 1].strip() or AL_ATTRIBUTE_RE.match(lines[start - 1])):
            start -= 1

        if start > starts[-1]:
            starts.append(start)

    starts.append(len(lines))
    sections = []
    for start, end in zip(starts, starts[1:]):
        section = "".join(lines[start:end]).strip()
        if section:
            sections.append(section)
    return sections or [content.strip()]

def chunk_oversized_section(record, section, is_file_end):
    prefix = f"{FILE_START_TOKEN}{record['source']}\n"
    suffix = f"\n{FILE_END_TOKEN}{EOS_TOKEN}" if is_file_end else ""
    prefix_tokens = backend_tokenizer.encode(prefix, add_special_tokens=False)
    suffix_tokens = backend_tokenizer.encode(suffix, add_special_tokens=False) if suffix else []
    available_tokens = CHUNK_TOKENS - len(prefix_tokens) - len(suffix_tokens)
    if available_tokens < MIN_CHUNK_TOKENS:
        raise ValueError(f"Chunk overhead is too large for {record['source']}")

    section_tokens = backend_tokenizer.encode(section, add_special_tokens=False)
    step = max(1, available_tokens - OVERLAP_TOKENS)
    chunks = []
    for start in range(0, len(section_tokens), step):
        token_slice = section_tokens[start:start + available_tokens]
        if len(token_slice) < MIN_CHUNK_TOKENS and chunks:
            break

        is_last_slice = start + available_tokens >= len(section_tokens)
        body_text = backend_tokenizer.decode(token_slice)
        chunks.append(chunk_text(record, body_text, is_file_end=is_file_end and is_last_slice, chunk_kind="token_window"))
        if is_last_slice:
            break
    return chunks

def chunk_file_record(record):
    file_text = format_file_record(record)
    token_ids = backend_tokenizer.encode(file_text + EOS_TOKEN, add_special_tokens=False)

    if len(token_ids) <= CHUNK_TOKENS:
        return [{"text": file_text + EOS_TOKEN, "source": record["source"], "is_file_end": True, "chunk_kind": "full_file"}]

    sections = split_al_sections(record["content"])
    chunks = []
    current_sections = []

    def flush_current(is_file_end=False):
        nonlocal current_sections
        if current_sections:
            chunks.append(chunk_text(record, "\n\n".join(current_sections), is_file_end=is_file_end))
            current_sections = []

    for section_index, section in enumerate(sections):
        is_last_section = section_index == len(sections) - 1
        candidate_sections = [*current_sections, section]
        candidate_fits = token_count(chunk_text(record, "\n\n".join(candidate_sections), is_file_end=is_last_section)["text"]) <= CHUNK_TOKENS

        if candidate_fits:
            current_sections = candidate_sections
            continue

        flush_current(is_file_end=False)
        section_fits = token_count(chunk_text(record, section, is_file_end=is_last_section)["text"]) <= CHUNK_TOKENS
        if section_fits:
            current_sections = [section]
        else:
            chunks.extend(chunk_oversized_section(record, section, is_file_end=is_last_section))

    flush_current(is_file_end=True)

    return chunks

if training_text_path.suffix.lower() == ".jsonl":
    file_records = parse_file_jsonl_records(raw_text)
else:
    file_records = parse_al_file_records(raw_text)
shuffled_file_records = file_records.copy()
random.Random(RANDOM_SEED).shuffle(shuffled_file_records)

eval_file_count = max(1, round(len(shuffled_file_records) * VALIDATION_FILE_RATIO)) if len(shuffled_file_records) > 1 else 0
eval_file_records = shuffled_file_records[:eval_file_count]
train_file_records = shuffled_file_records[eval_file_count:] or shuffled_file_records

def build_chunks(records):
    chunks = []
    chunks_per_file = []
    for record in records:
        record_chunks = chunk_file_record(record)
        chunks.extend(record_chunks)
        chunks_per_file.append(len(record_chunks))
    return chunks, chunks_per_file

train_chunks, train_chunks_per_file = build_chunks(train_file_records)
eval_chunks, eval_chunks_per_file = build_chunks(eval_file_records)

train_dataset = Dataset.from_list(train_chunks)
eval_dataset = Dataset.from_list(eval_chunks) if eval_chunks else None
dataset = Dataset.from_list(train_chunks + eval_chunks)

all_chunks = train_chunks + eval_chunks
all_chunks_per_file = train_chunks_per_file + eval_chunks_per_file
chunk_kind_counts = Counter(chunk["chunk_kind"] for chunk in all_chunks)
raw_token_count = sum(len(backend_tokenizer.encode(format_file_record(record), add_special_tokens=False)) for record in file_records)
print(f"Training text path: {training_text_path}")
print(f"Training text characters: {len(raw_text):,}")
print(f"Training text tokens: {raw_token_count:,}")
print(f"Source files: {len(file_records):,}")
print(f"Training files: {len(train_file_records):,}")
print(f"Validation files: {len(eval_file_records):,}")
print(f"Chunk tokens: {CHUNK_TOKENS:,}")
print(f"Overlap tokens inside long files: {OVERLAP_TOKENS:,}")
print(f"Files split into multiple chunks: {sum(count > 1 for count in all_chunks_per_file):,}")
print(f"Max chunks for one file: {max(all_chunks_per_file):,}")
print(f"Chunk kinds: {dict(chunk_kind_counts)}")
print(f"Training examples: {len(train_dataset):,}")
print(f"Validation examples: {len(eval_dataset) if eval_dataset is not None else 0:,}")
print(f"EOS token: {repr(EOS_TOKEN)}")

Let's verify the data by looking at a sample chunk:

In [ ]:
print(f"Sample chunk (first 500 chars):\n{'='*50}\n")
print(dataset[0]["text"][:500])
print(f"\n{'='*50}\n")
print(f"Sample chunk (middle of training text, first 500 chars):\n{'='*50}\n")
print(dataset[len(dataset)//2]["text"][:500])

For continued pretraining, we use the raw text directly without chat templates. The model will learn to predict the next token.

In [ ]:
# For continued pretraining, we just use the text as-is
# No chat template formatting needed - this is pure next-token prediction

# Let's see the first example
dataset[0]["text"][:200]

The AL text is already in the correct format for continued pretraining - we pass raw code text to the model and it learns to predict the next token.

In [ ]:
# Dataset preparation is file-aware now:
# - validation is split by whole AL files
# - chunks never cross file boundaries
# - EOS is appended only at true file ends
print(f"Dataset features: {dataset.features}")
print(f"Training examples: {len(train_dataset):,}")
print(f"Validation examples: {len(eval_dataset) if eval_dataset is not None else 0:,}")
print(f"EOS token: {repr(EOS_TOKEN)}")

The dataset is ready. Each example contains a chunk of text that will be used for next-token prediction training.

In [ ]:
# Look at an example
print(train_dataset[0]["text"][:300] + "...")

<a name="BaselineInference"></a>
### Baseline AL Completions Before Training

Run the fixed AL prompts before training so you can compare the base model against the continued-pretrained adapter later.


In [ ]:
# Fixed AL prompts for checking Business Central code completion quality.
test_prompts = [
    r"""tableextension 50100 CustomerExt extends Customer
{
    fields
    {
        field(50100; "Credit Risk Score"; Integer)
        {
            Caption = 'Credit Risk Score';""",
    r"""codeunit 50100 SalesOrderValidation
{
    procedure ValidateCustomerCreditLimit(SalesHeader: Record "Sales Header")
    var
        Customer: Record Customer;
    begin""",
    r"""pageextension 50100 CustomerCardExt extends "Customer Card"
{
    layout
    {
        addlast(General)
        {
            field("Credit Risk Score"; Rec."Credit Risk Score")""",
    r"""report 50100 "Customer Balance Export"
{
    UsageCategory = ReportsAndAnalysis;
    ApplicationArea = All;
    dataset
    {
        dataitem(Customer; Customer)""",
    r"""enum 50100 "BC Integration Status"
{
    Extensible = true;

    value(0; Pending)
    {
        Caption = 'Pending';""",
    r"""codeunit 50101 ItemAvailabilityMgt
{
    procedure GetAvailableInventory(ItemNo: Code[20]; LocationCode: Code[10]): Decimal
    var
        Item: Record Item;
    begin""",
]

FastModel.for_inference(model)

print("BASELINE COMPLETIONS BEFORE TRAINING")
print("=" * 50)
for prompt in test_prompts:
    print(f"\nPrompt:\n{prompt}")
    print("-" * 50)
    do_gemma_4_text_completion(prompt, max_new_tokens=192, temperature=0.2)

if hasattr(FastModel, "for_training"):
    FastModel.for_training(model)
else:
    model.train()


<a name="Train"></a>
### Train the model for Business Central Continued Pretraining

For continued pretraining, we train on all tokens. The goal is to adapt Gemma 4 E2B-it to AL syntax, Business Central object patterns, and Base Application coding conventions.

In [ ]:
import math
from trl import SFTTrainer, SFTConfig

# Stronger run: train for one full pass and keep the best validation checkpoint.
NUM_TRAIN_EPOCHS = 1.0
EVAL_STEPS = 25
PER_DEVICE_TRAIN_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
EFFECTIVE_BATCH_SIZE = PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
STEPS_PER_EPOCH = max(1, math.ceil(len(train_dataset) / EFFECTIVE_BATCH_SIZE))
SAVE_STEPS = EVAL_STEPS
WARMUP_STEPS = max(1, math.ceil(STEPS_PER_EPOCH * NUM_TRAIN_EPOCHS * 0.03))
print(f"Eval steps: {EVAL_STEPS}")
print(f"Save steps: {SAVE_STEPS}")
print(f"Warmup steps: {WARMUP_STEPS}")
print(f"Effective batch size: {EFFECTIVE_BATCH_SIZE}")

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_length = CHUNK_TOKENS + 1,
        packing = False,
        per_device_train_batch_size = PER_DEVICE_TRAIN_BATCH_SIZE,
        gradient_accumulation_steps = GRADIENT_ACCUMULATION_STEPS,
        warmup_steps = WARMUP_STEPS,
        num_train_epochs = NUM_TRAIN_EPOCHS,
        learning_rate = 5e-6,
        logging_steps = 10,
        eval_strategy = "steps",
        eval_steps = EVAL_STEPS,
        save_strategy = "steps",
        save_steps = SAVE_STEPS,
        save_total_limit = 3,
        load_best_model_at_end = True,
        metric_for_best_model = "eval_loss",
        greater_is_better = False,
        optim = "paged_adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        report_to = "none",
    ),
)

For continued pretraining, we train on ALL tokens (not just responses). We skip the `train_on_responses_only` step since we want the model to learn the full text distribution.

Let's verify the training setup by checking a tokenized example:

In [ ]:
# Tokenize and check the first training example
sample_text = train_dataset[0]["text"]
backend_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)
input_ids = backend_tokenizer.encode(sample_text, truncation=True, max_length=max_seq_length)
decoded = backend_tokenizer.decode(input_ids)

print(f"Original length: {len(sample_text)} chars")
print(f"Tokenized length: {len(input_ids)} tokens")
print(f"Trainer max_length: {CHUNK_TOKENS + 1} tokens")
print(f"\nFirst 200 chars of decoded:\n{decoded[:200]}...")

Verify that labels match the input_ids (next-token prediction):

In [ ]:
# Check that we're training on all tokens.
backend_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)
input_ids = backend_tokenizer.encode(train_dataset[0]["text"], truncation=True, max_length=max_seq_length)
print("Input IDs (first 10):", input_ids[:10])
print("Labels are the same sequence shifted internally for next-token prediction.")
print(f"\nFor continued pretraining, all {len(input_ids)} non-padding tokens in this chunk are used for training.")

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

# Let's train the model!

This stronger run uses `NUM_TRAIN_EPOCHS = 1.0`, evaluates and saves every 25 optimizer steps, and reloads the checkpoint with the lowest validation loss at the end.

To resume a training run from a saved checkpoint, set `trainer.train(resume_from_checkpoint = True)`.

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

<a name="Inference"></a>
### Inference for Business Central Continued Pretraining

For text completion inference after continued pretraining, use the model as a code completion model. The prompts below start AL objects or procedures and let the model continue them.

In [ ]:
# Prepare for post-training inference.
FastModel.for_inference(model)

print("POST-TRAINING COMPLETION")
print("=" * 50)
prompt = test_prompts[0]
print(f"Prompt:\n{prompt}")
print("-" * 50)
do_gemma_4_text_completion(prompt, max_new_tokens=256, temperature=0.2)

### Try more prompts

Test the model with different AL prompts to check whether it learned object structure, triggers, procedures, and Business Central naming patterns.

In [ ]:
# Test with different prompts.
for prompt in test_prompts:
    print(f"\nPrompt:\n{prompt}")
    print("-" * 50)
    do_gemma_4_text_completion(prompt, max_new_tokens=256, temperature=0.2)

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("gemma-4-e2b-it-bc-cpt")
tokenizer.save_pretrained("gemma-4-e2b-it-bc-cpt")
# model.push_to_hub("HF_ACCOUNT/gemma-4-e2b-it-bc-cpt", token = HF_TOKEN)
# tokenizer.push_to_hub("HF_ACCOUNT/gemma-4-e2b-it-bc-cpt", token = HF_TOKEN)

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastModel
    model, tokenizer = FastModel.from_pretrained(
        model_name = "gemma-4-e2b-it-bc-cpt",
        max_seq_length = MAX_SEQ_LENGTH,
        load_in_4bit = True,
    )

FastModel.for_inference(model)
do_gemma_4_text_completion(
    r"""codeunit 50100 CustomerPostingHelper
{
    procedure CheckPostingGroup(Customer: Record Customer)
    begin""",
    max_new_tokens = 512,
    temperature = 0.2,
)

### Saving to float16 for VLLM

You can also save the merged model for deployment. Set `if False` to `if True` when you want to export it.

In [ ]:
if False: # Change to True to save merged finetune.
    model.save_pretrained_merged("gemma-4-e2b-it-bc-cpt-merged", tokenizer)

If you want to upload / push to your Hugging Face account, set `if False` to `if True` and add your Hugging Face token and upload location!

In [ ]:
if False: # Change to True to upload merged finetune.
    model.push_to_hub_merged(
        "HF_ACCOUNT/gemma-4-e2b-it-bc-cpt-merged", tokenizer,
        token = HF_TOKEN
    )

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now for all models! For now, you can convert easily to `Q8_0, F16 or BF16` precision. `Q4_K_M` for 4bit will come later!

In [ ]:
if False: # Change to True to save to GGUF.
    model.save_pretrained_gguf(
        "gemma-4-e2b-it-bc-cpt-gguf",
        tokenizer,
        quantization_method = "Q8_0",
    )

Likewise, if you want to instead push to GGUF to your Hugging Face account, set `if False` to `if True` and add your Hugging Face token and upload location!

In [ ]:
if False: # Change to True to upload GGUF.
    model.push_to_hub_gguf(
        "HF_ACCOUNT/gemma-4-e2b-it-bc-cpt-gguf",
        tokenizer,
        quantization_method = "Q8_0",
        token = HF_TOKEN,
    )